# Sprint 007 D2A — Midpoint-to-cross dollar-P&L bridge

**Question:** Of the accepted midpoint-to-cross dollar-P&L shortfall, how much comes from direct entry-price concession, how much from fill-dependent sizing and financing, and where is the dominant mechanism concentrated?

This notebook is **D2A only**. It stops at the human checkpoint: aggregate, side, and yearly dollar bridge, one order-sensitivity statistic, one D2B branch recommendation, and a **provisional** D3 class.

**Sign convention:** all terms are dollars of P&L (profit > 0).

```
G       = P_cross − P_mid
Δ_price = P(Q_mid, p_cross) − P(Q_mid, p_mid)
Δ_size  = P(Q_cross, p_cross) − P(Q_mid, p_cross)
Δ_set   = P_cross_unmatched − P_mid_unmatched
R       = G − (Δ_price + Δ_size + Δ_set)
```

Formulas are frozen in [`docs/tmp/sprint007_d2_design.md`](../../docs/tmp/sprint007_d2_design.md).

**Non-goals:** D2B diagnostics, D3 design, fill ladders, filters, alternative-structure P&L, yearly return/CAR tables, `SurfaceRunner`.

In [ ]:
import sys
from pathlib import Path


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "src" / "backtest").is_dir() and (candidate / "setup.py").exists():
            return candidate
    raise RuntimeError("Could not locate MomentumCVG repo root (set cwd or PYTHONPATH)")


REPO_ROOT = _repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.backtest.sprint007_artifact_validation import run_d0_validation
from src.backtest.sprint007_d1_gross_margin import VERDICT_CONTINUE, run_d1_analysis
from src.backtest.sprint007_d2_shortfall_bridge import (
    VERDICT_BLOCKED,
    resolve_evidence_dir,
    run_d2a_analysis,
    write_d2a_tables,
)

pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
plt.rcParams["figure.figsize"] = (9, 4)

## 1. D0 readiness and D1 continue (accepted / blocker)

D0 must pass and D1 must be `D1_CONTINUE_TO_D2`. If either fails, D2A is `D2_BLOCKED` and this notebook stops before interpreting the gap.

In [ ]:
d0 = run_d0_validation()
print("D0 verdict:", d0.verdict)
print("D0 gates all passed:", d0.all_passed)
for gate in d0.gates:
    status = "PASS" if gate.passed else "FAIL"
    print(f"{gate.gate_id} [{status}] {gate.detail}")

d1 = run_d1_analysis(d0_result=d0)
print("D1 verdict:", d1.verdict)

result = run_d2a_analysis(d0_result=d0, d1_result=d1)
print("D2A blocked:", result.blocked)
print("D2A verdict:", result.verdict)
if result.blocked or d1.verdict != VERDICT_CONTINUE:
    raise RuntimeError(
        "D2A blocked — do not interpret shares or recommend D2B: "
        f"{result.blocker}"
    )

## 2. Joins, leg-to-trade, and Δ_set (accepted calculation)

Included keys are joined on `(trade_date, ticker, direction)`. Unmatched keys are a blocker, not an attribution target.

In [ ]:
integrity = pd.DataFrame(
    [{"name": item.name, "passed": item.passed, "detail": item.detail} for item in result.integrity]
)
recon = pd.DataFrame(result.reconciliation)
display(integrity)
display(recon)
print("unmatched mid/cross keys:", result.bridge.n_mid_only, result.bridge.n_cross_only)
print("Δ_set:", result.bridge.delta_set)

## 3. Aggregate waterfall (accepted calculation)

`P_mid → Δ_price → Δ_size → Δ_set → R → P_cross`. Residual must sit inside the frozen dollar tolerance.

In [ ]:
b = result.bridge
agg = pd.Series(
    {
        "P_mid": b.p_mid,
        "Δ_price": b.delta_price,
        "Δ_size": b.delta_size,
        "Δ_set": b.delta_set,
        "R": b.residual,
        "P_cross": b.p_cross,
        "G": b.gap,
    },
    name="dollars",
).to_frame()
display(agg)

steps = ["P_mid", "Δ_price", "Δ_size", "Δ_set", "R"]
values = [b.p_mid, b.delta_price, b.delta_size, b.delta_set, b.residual]
running = [0.0]
for value in values[:-1]:
    running.append(running[-1] + value)

fig, ax = plt.subplots()
colors = ["#1f77b4"] + ["#d62728" if v < 0 else "#2ca02c" for v in values[1:]]
ax.bar(steps, values, bottom=running, color=colors, width=0.65)
ax.axhline(0.0, color="#888888", linewidth=0.8)
ax.axhline(b.p_cross, color="#4c72b0", linewidth=1.0, linestyle="--", label="P_cross")
ax.set_title("D2A dollar bridge — Laspeyres / Q_mid")
ax.set_ylabel("pnl_total ($)")
ax.legend()
plt.show()

## 4. Long versus short (accepted calculation)

Same identity by `direction`. Side-split `Δ_size` is date-coupled (longs are financed by short credit) and is not an independent long-side spread tax.

In [ ]:
sides = pd.DataFrame(result.side_bridge)
display(sides)

fig, ax = plt.subplots(figsize=(7, 3.5))
x = np.arange(len(sides))
width = 0.35
ax.bar(x - width / 2, sides["delta_price"], width, label="Δ_price", color="#d62728")
ax.bar(x + width / 2, sides["delta_size"], width, label="Δ_size", color="#ff7f0e")
ax.axhline(0.0, color="#888888", linewidth=0.8)
ax.set_xticks(x, sides["slice"])
ax.set_title("Dollar bridge components by side")
ax.set_ylabel("dollars")
ax.legend()
plt.show()

## 5. Yearly dollar bridge (accepted calculation)

Calendar-year **dollar** components of `G`. This is not a yearly return or CAR table (that remains a D1 reporting item).

In [ ]:
years = pd.DataFrame(result.yearly_bridge)
display(years)

fig, ax = plt.subplots()
ax.bar(years["slice"], years["delta_price"], label="Δ_price", color="#d62728")
ax.bar(
    years["slice"],
    years["delta_size"],
    bottom=years["delta_price"],
    label="Δ_size",
    color="#ff7f0e",
)
ax.axhline(0.0, color="#888888", linewidth=0.8)
ax.set_title("Yearly stacked Δ_price + Δ_size")
ax.set_xlabel("year")
ax.set_ylabel("dollars")
ax.legend()
plt.show()

## 6. Order-sensitivity companion (exploratory description)

One statistic: `S_order = |I| / |G|`, identical to the Laspeyres–Paasche price difference. It is attribution sensitivity only. It does **not** select the D2B branch. It sets `order_sensitive` only if the dual order would change materiality or dominance.

In [ ]:
cls = result.classification
sens = pd.Series(
    {
        "S_order": b.s_order,
        "interaction": b.interaction,
        "Δ_price": b.delta_price,
        "Δ_price_Paasche": b.delta_price_paasche,
        "Δ_size": b.delta_size,
        "Δ_size_dual": b.delta_size_dual,
        "price_material": cls.price_material,
        "size_material": cls.size_material,
        "dominant": cls.dominant,
        "order_sensitive": cls.order_sensitive,
    },
    name="value",
).to_frame()
display(sens)

## 7. Human checkpoint — D2B branch and provisional D3 class

Review the residual, `Δ_set`, side/year tables, and `S_order` before authorizing D2B. This cell does **not** run D2B, assign a final D3 class, or design D3.

In [ ]:
checkpoint = pd.Series(
    {
        "D2B branch": cls.d2b_branch,
        "provisional D3 class": cls.provisional_d3_class,
        "final D3 class": None,
        "concentrating side": cls.concentrating_side,
        "structure": cls.structure,
    },
    name="checkpoint",
).to_frame()
display(checkpoint)
print(result.conclusion)
print("STOP: do not run D2B or assign a final D3 class until this checkpoint is accepted.")

EVIDENCE_DIR = resolve_evidence_dir()
write_d2a_tables(result, EVIDENCE_DIR)
print("evidence dir:", EVIDENCE_DIR)